# Hand-Mechanism 3D Interaction Analysis

**Question:** When a lockbox mechanism changes state, is the mouse's paw actually *near* that
mechanism at that moment, and which paw, and how far in advance does it arrive?

This predicts **nothing**. It measures the spatial-temporal coincidence of two reconstructions
we obtained independently in the *same* metric 3D frame:
- mouse paws (transfer-finetuned DLC to triangulated 3D)
- lockbox mechanisms (DLC trained on 20% of scene 1 to triangulated 3D)

**Outputs**
- `distance_timeseries.csv`, per-frame paw-mechanism distances (mm)
- `contact_events.csv`, contact onsets vs GT state-change onsets, with lead/lag in seconds
- `interaction_summary.csv`, one row per mechanism: which paw, lead time, did contact precede change
- plots: distance curves with state-change markers. Lead-time summary. Reach raster

---

## 0. Config (the only two things you must edit)

1. `PAW_KPS`, your mouse DLC bodypart names that count as "hands".
2. `MECH_KPS`, which lockbox 3D keypoint(s) define each mechanism's position. If a mechanism
   maps to several keypoints, we use their centroid per frame.

Everything else (paths, fps, thresholds) is below and self-explanatory.

In [ ]:
import pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Paths
LOCKBOX_STATE_PKL  = Path(r'../data/lockbox_state/scene1/lockbox_state.pkl').resolve()
PIPELINE_STATE_PKL = Path('../data/triangulate_render/scene1/pipeline_state.pkl').resolve()
OUT_DIR            = Path(r'../data/hand_mechanism_interaction/scene1').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

FPS = 30

PAW_KPS = ['front_left_paw', 'front_right_paw', 'nose']   # <-- set to your actual bodypart names

# EDIT: map each mechanism 
MECH_KPS = {
    'lever1':  ['lever1_push_end'],
    'slider1': ['slider1_knob'],
    'ball1':   ['ball1_center'],
    'cover1':  ['cover_marker'],
}

# Contact / analysis params 
CONTACT_MM = 30         # paw within this distance (mm) of a mechanism counts as 'contact'
MIN_CONTACT_FR = 2      # require this many consecutive in-contact frames to call a contact onset
SMOOTH_FR      = 5      # median-smooth distance curves over this many frames (0 = off)
PRE_WINDOW_S   = 6.0    # search this many seconds BEFORE a state change for the contact onset

print('Config loaded. Edit PAW_KPS and MECH_KPS to match your bodypart names if needed.')

Config loaded. Edit PAW_KPS and MECH_KPS to match your bodypart names if needed.


## 1. Load both 3D reconstructions

In [ ]:
with open(LOCKBOX_STATE_PKL, 'rb') as f:
    lk = pickle.load(f)
with open(PIPELINE_STATE_PKL, 'rb') as f:
    ms = pickle.load(f)

stage       = lk['stage'].astype(np.float32)          # (n_frames,) monotone 0..4
STAGE_NAMES = lk['STAGE_NAMES']
STAGE_ORDER = lk['STAGE_ORDER']                        # ['lever1','slider1','ball1','cover1']
lk_3d       = lk['points_3d']                          # (n_frames, n_lk, 3) mm, NaN where missing
lk_valid    = list(lk['valid_kps'])

ms_3d       = ms['points_3d']                          # (n_frames_m, n_ms, 3) mm, NaN where missing
ms_valid    = list(ms['valid_kps'])

# Align frame counts
n_frames = min(lk_3d.shape[0], ms_3d.shape[0])
stage, lk_3d, ms_3d = stage[:n_frames], lk_3d[:n_frames], ms_3d[:n_frames]
t = np.arange(n_frames) / FPS

# Resolve keypoint indices, keeping only those that actually exist
paw_idx = {bp: ms_valid.index(bp) for bp in PAW_KPS if bp in ms_valid}
missing_paws = [bp for bp in PAW_KPS if bp not in ms_valid]

mech_idx = {}
for mech, kps in MECH_KPS.items():
    kps = [kps] if isinstance(kps, str) else kps
    present = [k for k in kps if k in lk_valid]
    if present:
        mech_idx[mech] = [lk_valid.index(k) for k in present]

print(f'n_frames                 : {n_frames}  ({t[-1]:.1f}s @ {FPS}fps)')
print(f'paws resolved            : {list(paw_idx)}')
if missing_paws:
    print(f'  PAW_KPS not found in mouse data: {missing_paws}')
print(f'mechanisms resolved      : {list(mech_idx)}')
for m in STAGE_ORDER:
    if m not in mech_idx:
        print(f'  mechanism {m!r} has no resolvable lockbox keypoint — check MECH_KPS')
print(f'mouse 3D coverage        : {np.isfinite(ms_3d).all(-1).mean()*100:.1f}%')
print(f'lockbox 3D coverage      : {np.isfinite(lk_3d).all(-1).mean()*100:.1f}%')

n_frames                 : 8190  (273.0s @ 30fps)
paws resolved            : ['front_left_paw', 'front_right_paw', 'nose']
mechanisms resolved      : ['lever1', 'slider1', 'ball1', 'cover1']
mouse 3D coverage        : 58.6%
lockbox 3D coverage      : 82.8%


## 2. Paw-mechanism distance timeseries (mm)

For every (paw, mechanism) pair we compute the per-frame 3D Euclidean distance. Mechanism
position is the centroid of its mapped keypoints. Frames where either point is missing are NaN
(an honest gap, not a zero).

In [3]:
def mech_position(mech):
    """(n_frames, 3) centroid of the mechanism's lockbox keypoints (NaN-aware)."""
    pts = lk_3d[:, mech_idx[mech], :]                 # (n_frames, k, 3)
    return np.nanmean(pts, axis=1)                    # NaN only if all kps missing that frame

def median_smooth(x, w):
    if not w or w < 2:
        return x
    out = x.copy(); h = w // 2
    for i in range(len(x)):
        seg = x[max(0, i-h):i+h+1]
        seg = seg[np.isfinite(seg)]
        if seg.size:
            out[i] = np.median(seg)
    return out

dist = {}  # (paw, mech) -> (n_frames,) mm
for mech in mech_idx:
    mp = mech_position(mech)
    for paw, pi in paw_idx.items():
        pp = ms_3d[:, pi, :]
        d  = np.sqrt(((pp - mp) ** 2).sum(axis=1))    # NaN-propagating, intended
        dist[(paw, mech)] = median_smooth(d, SMOOTH_FR)

# Flat tidy CSV
rows = []
for (paw, mech), d in dist.items():
    for fr in range(n_frames):
        rows.append(dict(frame=fr, t_s=t[fr], paw=paw, mechanism=mech,
                         dist_mm=d[fr], stage=stage[fr]))
dist_df = pd.DataFrame(rows)
dist_df.to_csv(OUT_DIR / 'distance_timeseries.csv', index=False)
print(f'distance_timeseries.csv  : {len(dist_df)} rows '
      f'({len(dist)} paw×mech pairs × {n_frames} frames)')
print('median dist per pair (mm):')
print(dist_df.groupby(['mechanism','paw'])['dist_mm'].median().round(1))

distance_timeseries.csv  : 98280 rows (12 paw×mech pairs × 8190 frames)
median dist per pair (mm):
mechanism  paw            
ball1      front_left_paw      60.6
           front_right_paw     55.4
           nose                72.9
cover1     front_left_paw      64.6
           front_right_paw     55.4
           nose                39.3
lever1     front_left_paw      92.8
           front_right_paw     91.5
           nose                82.3
slider1    front_left_paw      92.1
           front_right_paw     88.4
           nose               105.2
Name: dist_mm, dtype: float64


## 3. State-change onsets and contact onsets

- **State-change onset** for mechanism *k*: first frame the monotone `stage` reaches *k* (the
  moment that mechanism is solved). This is your GT timing.
- **Contact onset**: first frame, within `PRE_WINDOW_S` *before* the state change, where the
  nearer paw is within `CONTACT_MM` for at least `MIN_CONTACT_FR` consecutive frames.

We report **lead time** = state-change time − contact onset time. Positive lead = paw arrived
*before* the mechanism changed (the sensible direction).

In [4]:
def state_change_onsets(stage_curve, n_stages):
    s = np.maximum.accumulate(np.nan_to_num(stage_curve, nan=-np.inf))
    out = {}
    for k in range(1, n_stages):
        hit = np.where(s >= k - 0.5)[0]
        out[k] = int(hit[0]) if len(hit) else None
    return out

def first_contact_run(d, start, end, thr, min_run):
    """First frame in [start,end) beginning a run of >=min_run consecutive frames with d<thr."""
    in_c = (d < thr)
    run = 0
    for fr in range(max(0, start), min(len(d), end)):
        run = run + 1 if (np.isfinite(d[fr]) and in_c[fr]) else 0
        if run >= min_run:
            return fr - min_run + 1
    return None

sc = state_change_onsets(stage, len(STAGE_NAMES))   # {stage_k -> frame}
pre_fr = int(PRE_WINDOW_S * FPS)

event_rows = []
for k in range(1, len(STAGE_NAMES)):
    mech = STAGE_ORDER[k-1] if k-1 < len(STAGE_ORDER) else None
    sc_fr = sc.get(k)
    if mech not in mech_idx or sc_fr is None:
        continue
    win_start = sc_fr - pre_fr
    # Allow a little slack after the change too, in case contact is detected slightly late
    win_end   = sc_fr + pre_fr
    for paw in paw_idx:
        d = dist[(paw, mech)]
        c_fr = first_contact_run(d, win_start, win_end, CONTACT_MM, MIN_CONTACT_FR)
        min_d_in_win = np.nanmin(d[max(0,win_start):win_end]) if win_end > 0 else np.nan
        event_rows.append(dict(
            stage=k, mechanism=mech, paw=paw,
            state_change_s=sc_fr / FPS,
            contact_onset_s=(c_fr / FPS) if c_fr is not None else np.nan,
            lead_time_s=((sc_fr - c_fr) / FPS) if c_fr is not None else np.nan,
            min_dist_in_window_mm=float(min_d_in_win),
            contact_detected=c_fr is not None,
        ))
events_df = pd.DataFrame(event_rows)
events_df.to_csv(OUT_DIR / 'contact_events.csv', index=False)
print('contact_events.csv:')
print(events_df.to_string(index=False))

contact_events.csv:
 stage mechanism             paw  state_change_s  contact_onset_s  lead_time_s  min_dist_in_window_mm  contact_detected
     1    lever1  front_left_paw      179.766667       178.533333     1.233333              14.306250              True
     1    lever1 front_right_paw      179.766667              NaN          NaN              30.159993             False
     1    lever1            nose      179.766667       178.366667     1.400000              16.692015              True
     2   slider1  front_left_paw      186.666667              NaN          NaN              80.246384             False
     2   slider1 front_right_paw      186.666667              NaN          NaN              75.466992             False
     2   slider1            nose      186.666667              NaN          NaN              80.964253             False
     3     ball1  front_left_paw      208.833333              NaN          NaN              36.036505             False
     3     ball1 fro

## 4. Per-mechanism interaction summary

For each mechanism: which paw got closest, the lead time of that paw, and whether contact
preceded the state change. This is the table that answers "does the mouse touch what it solves?"

In [5]:
summ_rows = []
for mech in [m for m in STAGE_ORDER if m in mech_idx]:
    sub = events_df[events_df.mechanism == mech]
    if sub.empty:
        continue
    # nearest paw = smallest min distance in the window
    best = sub.loc[sub['min_dist_in_window_mm'].idxmin()]
    summ_rows.append(dict(
        mechanism=mech,
        nearest_paw=best['paw'],
        min_dist_mm=round(float(best['min_dist_in_window_mm']), 1),
        contact_detected=bool(best['contact_detected']),
        lead_time_s=(round(float(best['lead_time_s']), 2)
                     if np.isfinite(best['lead_time_s']) else None),
        contact_precedes_change=(bool(best['lead_time_s'] >= 0)
                                 if np.isfinite(best['lead_time_s']) else None),
        state_change_s=round(float(best['state_change_s']), 2),
    ))
summary_df = pd.DataFrame(summ_rows)
summary_df.to_csv(OUT_DIR / 'interaction_summary.csv', index=False)
print('interaction_summary.csv:')
print(summary_df.to_string(index=False))

n_ok = summary_df['contact_precedes_change'].fillna(False).sum()
n_tot = len(summary_df)
print(f'\n-> contact preceded state change for {n_ok}/{n_tot} mechanisms')

interaction_summary.csv:
mechanism     nearest_paw  min_dist_mm  contact_detected  lead_time_s contact_precedes_change  state_change_s
   lever1  front_left_paw         14.3              True         1.23                    True          179.77
  slider1 front_right_paw         75.5             False          NaN                    None          186.67
    ball1 front_right_paw         26.6              True        -0.13                   False          208.83
   cover1            nose         20.9              True         0.67                    True          211.43

-> contact preceded state change for 2/4 mechanisms


## 5. Plots

### 5a. Distance curves with state-change markers
One panel per mechanism. Solid lines = paw distances. Dashed vertical = GT state change;
dotted horizontal = contact threshold. Dot = detected contact onset.

In [6]:
mechs_present = [m for m in STAGE_ORDER if m in mech_idx]
fig, axes = plt.subplots(len(mechs_present), 1,
                          figsize=(13, 2.4*len(mechs_present)), sharex=True)
if len(mechs_present) == 1:
    axes = [axes]
paw_colors = {p: c for p, c in zip(paw_idx, ['tab:blue','tab:red','tab:green','tab:purple'])}

for ax, mech in zip(axes, mechs_present):
    for paw in paw_idx:
        ax.plot(t, dist[(paw, mech)], lw=1.0, color=paw_colors[paw], label=paw, alpha=0.85)
    ax.axhline(CONTACT_MM, ls=':', color='gray', lw=1, label=f'contact={CONTACT_MM:.0f}mm')
    k = STAGE_ORDER.index(mech) + 1
    if sc.get(k) is not None:
        ax.axvline(sc[k]/FPS, ls='--', color='k', lw=1.3, label='state change')
    sub = events_df[(events_df.mechanism == mech) & (events_df.contact_detected)]
    for _, r in sub.iterrows():
        ax.plot(r['contact_onset_s'], CONTACT_MM, 'o', color=paw_colors[r['paw']], ms=7, zorder=5)
    ax.set_ylabel(f'{mech}\ndist [mm]', fontsize=9)
    ax.legend(fontsize=7, ncol=4, loc='upper right', frameon=False)
    ax.grid(alpha=0.25)
axes[-1].set_xlabel('time [s]')
fig.suptitle('Paw–mechanism distance with GT state-change markers', y=1.002, fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'distance_curves.png', dpi=120, bbox_inches='tight')
plt.show()

### 5b. Lead-time summary
Positive bar = paw reached the mechanism *before* it changed state (the validating direction).

In [7]:
plot_df = events_df[events_df.contact_detected].copy()
if not plot_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    labels = [f"{r.mechanism}\n{r.paw}" for r in plot_df.itertuples()]
    vals = plot_df['lead_time_s'].values
    bars = ax.bar(range(len(vals)), vals,
                  color=['tab:green' if v >= 0 else 'tab:red' for v in vals])
    ax.axhline(0, color='k', lw=1)
    ax.set_xticks(range(len(vals))); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('lead time [s]  (paw before state change)')
    ax.set_title('How early did the paw arrive at each mechanism?')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'lead_time.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No contacts detected within the window — see the diagnostics in section 6.')